# Demostration of ***Jupyphant***
### a jupyterlab extenstion for interactive data analysis, exploration and visualization

## Getting Started

In order to start Jupyphant, open the commands pallete on the left side (loupe-sympbole or Crtl+Shift+C) and   
search for the 'Jupyphant' command in the 'NeuroSciece' category.  
Then click it and the 'Jupyphant' tab will open in the foreground.  

In the upper half of the splitted panel, an iptree of neo-objects will be displayed on the left side and the node explorer on the right side.  
The ipytree of neo-objects follows the neo-hierarchy in order to relate child-objects to parent-containers.  
If you select a node (by clicking) that starts with 'XXX::' you can see metadata information in the 'INFO' tab of the node explorer.  
In case the selected node represents a neo data objekt like 'AnalogSignal' (ASG) or 'SpikeTrain' (SPT),  
then the node explorer shows in the 'RAW PLOT' tab the raw plots the selected nodes and  
in the 'STATISTICS' tab some basic statistics of the respective selected nodes like  
ISI-distribution, time-histgram, IFR, correlation-matrix for 'SpikeTrains'.

In the lower half of the splitted panel, overview rasterplots for 'SpikeTrains' and LFP-plots for 'AnalogSignals' are shown each top-node of the ipytree from the upper half.



In [1]:
from neo import AnalogSignal, Event, Block, SpikeTrain, Segment, Epoch, IrregularlySampledSignal, ImageSequence, Group, CircularRegionOfInterest, RectangularRegionOfInterest, PolygonRegionOfInterest, ChannelView
from neo.io.blackrockio import BlackrockIO
import numpy as np
import quantities as pq

In [ ]:
# Read complete block which should be displayed as a TreeView by Jupyphant
dirname = 'l101210-001'
reader = BlackrockIO(dirname, nsx_to_load=2)
block = reader.read_block(load_waveforms=False, signal_group_mode="split-all")

In [ ]:
spiketrains = []
# each segment contains a single trial
for ind in range(len(block.segments)):
    spiketrains.append(block.segments[ind].spiketrains)
spiketrainsSTL = spiketrains[0]

In [ ]:
# Create independent Objects that should be displayed as well
sptr = SpikeTrain([1,2,3], name="SpikeTrain 1", t_stop=4, units='s', id='Unit 1000', channel_id=1, unit_id= 0, unit_tag='unclassified')
anasig = AnalogSignal([[1, 2, 3], [4, 5, 6]], units='V', sampling_rate=1*pq.Hz, name="AnalogSignal 1")

In [ ]:
# small block with one segment, one analogsignal and one spiketrain
block2 = Block(name="my block") 
block2.segments.append(Segment(name="my segment"))
block2.segments[0].analogsignals.append(AnalogSignal([1,2,3], name="my analogsignal", t_stop=4, units='s', sampling_rate=1*pq.Hz, id='Unit 1', channel_id=1, unit_id=0, unit_tag='unclassified'))
block2.segments[0].spiketrains.append(SpikeTrain([1,2,3], name="my spiketrain", t_stop=4, units='s', id='Unit 1', channel_id=1, unit_id=0, unit_tag='unclassified'))

In [ ]:
evt = Event([1,2,3]*pq.ms, labels=['1', '2', '3'], name="Event 1")
epc = Epoch(times=np.arange(0, 30, 10)*pq.s, durations=[10, 5, 7]*pq.ms, labels=np.array(['btn0', 'btn1', 'btn2'], dtype='U'), name="Epoch 1")

In [ ]:
irrsig = IrregularlySampledSignal([0.0, 1.23, 6.78], [1, 2, 3], units='mV', time_units='ms', name="IrregularlySampledSignal 1")
imgseq = ImageSequence([[[column for column in range(20)]for row in range(20)] for frame in range(10)], units='V', sampling_rate=1 * pq.Hz, spatial_scale=1 * pq.micrometer, name="ImageSequence 1") 

In [ ]:
rroi = RectangularRegionOfInterest(20.0, 20.0, width=5.0, height=5.0)
croi = CircularRegionOfInterest(20.0, 20.0, radius=5.0)
proi = PolygonRegionOfInterest((20.0, 20.0), (30.0, 20.0), (25.0, 25.0))        

In [ ]:
chlv = ChannelView(anasig, index=[0, 1, 2], name="Channel Group 1")

In [ ]:
seg = Segment(index=1, name="Segment 1")
seg.spiketrains.append(sptr)
seg.analogsignals.append(anasig)
seg.epochs.append(epc)
seg.events.append(evt)
seg.irregularlysampledsignals.append(irrsig)
seg.imagesequences.append(imgseq)

In [ ]:
grp = Group(name="Group 1")
grp.analogsignals.append(anasig)
grp.spiketrains.append(sptr)
grp.irregularlysampledsignals.append(irrsig)
grp.segments.append(Segment(index=2, name="Segment 2"))
grp.groups.append(Group(name="Group 2"))

In [ ]:
blk = Block(name="Block 1")
blk.segments.append(seg)
blk.groups.append(grp)
blk.regionsofinterest += [rroi, croi, proi]

In [ ]:
# List of independent objects (independent references), should be displayed as well
sptrs = block.list_children_by_class(SpikeTrain)

In [ ]:
# Not supported yet
# Nested list of independent objects (independent references)
sptrs.append([sptr])
nested_sptrs = [[sptrs[0]]]

# Experimental Area

## Debugging

For debugging only: The Python object created by the extension is called `jupyphant_entity`

In [ ]:
# Examplary access to the extension's Python object and its methods
jupyphant_entity.ipytree_of_neo_objects

## Benchmarking

In [ ]:
from ipytree import Tree, Node
from IPython.display import display

import time

In [ ]:
tree = Tree()
tree.stripes=True
trunk = Node(f'trunk')
trunk.opened = False
tree.add_node(trunk)


subnodes_per_node =5
levels =5

def create_nested_tree(curr_node, levels, nodes_per_level):
    if levels > 0:
        for i in range(nodes_per_level):
            node = Node(f'branch_lv{levels}_twig{i}')
            node.opened = False
            create_nested_tree(node, levels-1, nodes_per_level)
            curr_node.add_node(node)
            
create_nested_tree(trunk, levels, subnodes_per_node)

total_num_of_nodes = 1
print(f"Number of nodes:\nlevel 0: 1")
for l in range(levels):
    num_nodes_on_same_level = subnodes_per_node ** (l+1)
    total_num_of_nodes += num_nodes_on_same_level
    print(f"level {l+1}: {num_nodes_on_same_level}")
print(f"Total number of nodes: {total_num_of_nodes}")

tree

In [ ]:
# static function for node counting
def count_nodes(tree, n_nodes=None):
    if n_nodes is None:
        n_nodes = 0
    curr_node = tree
    for n in curr_node.nodes:
        n_nodes += 1
        if len(n.nodes) > 0:       
            n_nodes = count_nodes(n, n_nodes)
        
    return n_nodes

n_nodes = 0
t_start = time.time()
print(f"counted nodes: {count_nodes(tree, n_nodes)}")
t_stop = time.time()
print(f"time: {t_stop-t_start} s")

In [ ]:
# oop for node counting
class CountNodes:
    def __init__(self, tree):
        self.tree = tree
        if isinstance(self.tree, Tree):
            self.n_nodes = 0
        elif  isinstance(self.tree, Node):
            self.n_nodes = 1
        else:
            raise TypeError("tree-parameter must be an instance of Tree or Node")
        self.n_nodes = self.count_nodes(self.tree)

    def count_nodes(self, curr_node):
        for n in curr_node.nodes:
            self.n_nodes += 1
            if len(n.nodes) > 0:       
                self.n_nodes = self.count_nodes(n)

        return self.n_nodes

In [ ]:
t_start = time.time()
print(f"counted nodes: {CountNodes(tree).n_nodes}")
t_stop = time.time()
print(f"time: {t_stop-t_start} s")

In [ ]:
for tree in jupyphant_entity.ipytree_of_neo_objects.nodes:
    n_nodes = 0
    print(f"{tree.name}: n_nodes = {CountNodes(tree).n_nodes}")